<a href="https://colab.research.google.com/github/Aigulya0/project-03-income/blob/main/Project_03_Income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект 3 — Уровень дохода
**Модуль 1, часть Б | Базовый + Основной + Продвинутый**

Задача: предсказать, превышает ли годовой доход человека $50K. Учебный пример; не использовать для реальных кредитных решений.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
print('Библиотеки загружены')

## 1. Загрузка данных
В Colab загрузите `adult.csv` через значок папки слева → Upload. При локальном запуске файл берётся из `data/adult.csv`.

In [ ]:
paths = [Path('adult.csv'),Path('/content/adult.csv'),Path('data/adult.csv'),Path('../data/adult.csv')]
path = next((p for p in paths if p.exists()), None)
if path is None:
    from google.colab import files
    uploaded = files.upload()
    path = Path(next(iter(uploaded)))
df = pd.read_csv(path, skipinitialspace=True)
print('Размер:',df.shape)
display(df.head())

## 2. EDA: типы, пропуски, дубликаты, распределения

In [ ]:
print(df.info()); display(df.describe(include='all').T)
print('Дубликаты:',df.duplicated().sum())
print('Пропуски до очистки:'); display(df.isna().sum().sort_values(ascending=False))
print('Скрытые пропуски "?":'); display(df.astype(str).eq('?').sum().sort_values(ascending=False))
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)
print('Пропуски после очистки:'); display(df.isna().sum().sort_values(ascending=False))
print('Баланс классов:'); display(df['income'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,4))
sns.countplot(data=df,x='income',ax=ax[0]);ax[0].set_title('Доход')
sns.histplot(data=df,x='age',hue='income',bins=25,ax=ax[1]);ax[1].set_title('Возраст')
sns.boxplot(data=df,x='income',y='hours.per.week',ax=ax[2]);ax[2].set_title('Рабочие часы')
plt.tight_layout();plt.show()
plt.figure(figsize=(9,6));sns.heatmap(df.select_dtypes('number').corr(),annot=True,cmap='coolwarm',fmt='.2f');plt.title('Корреляции');plt.show()
print('Доли дохода >50K по группам (описательный анализ, не причинная связь):')
for c in ['sex','race','education']:
    print('\n',c);display(pd.crosstab(df[c],df['income'],normalize='index').round(3))

## 3. Разделение и обработка без утечки
Разбиваем 80/20 со стратификацией и seed=42. `education.num` удаляем как дублирование `education`, `fnlwgt` — как вес переписи, не прямую характеристику дохода. Все импьютеры, кодировщики и масштабирование обучаются только на train внутри Pipeline.

In [ ]:
X = df.drop(columns=['income','education.num','fnlwgt']).copy()
y = (df['income'].astype(str).str.strip()=='>50K').astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
num=X_train.select_dtypes(include='number').columns.tolist()
cat=X_train.select_dtypes(exclude='number').columns.tolist()
print('Числовые:',num,'\nКатегориальные:',cat)
print('Train:',X_train.shape,'Test:',X_test.shape)
def preprocessor():
    return ColumnTransformer([
        ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
        ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)
    ])
models={
 'Logistic Regression':Pipeline([('prep',preprocessor()),('model',LogisticRegression(max_iter=1500,random_state=42))]),
 'Random Forest':Pipeline([('prep',preprocessor()),('model',RandomForestClassifier(n_estimators=120,min_samples_leaf=3,random_state=42,n_jobs=-1))])
}

## 4. StratifiedKFold(5): сравнение моделей
Метрики: accuracy, precision, recall, F1, ROC-AUC. Проверка порога и выбор модели — только на train.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,model in models.items():
    scores=cross_validate(model,X_train,y_train,cv=cv,scoring={'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'},n_jobs=-1)
    row={'model':name}
    for m in ['accuracy','precision','recall','f1','roc_auc']:
        row[m]=round(scores['test_'+m].mean(),4)
    rows.append(row)
results=pd.DataFrame(rows).sort_values('roc_auc',ascending=False)
display(results)
best_name=results.iloc[0]['model'];best_model=models[best_name]
print('Лучшая по CV ROC-AUC:',best_name)

## 5. Порог по стоимости ошибок
По условию ошибки сопоставимы, поэтому демонстрируем равные условные стоимости: FP=1, FN=1. Порог выбирается по out-of-fold вероятностям train; тест остаётся нетронутым.

In [ ]:
oof=cross_val_predict(best_model,X_train,y_train,cv=cv,method='predict_proba',n_jobs=-1)[:,1]
thresholds=np.arange(.10,.91,.05)
threshold_table=[]
for t in thresholds:
    pred=(oof>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_train,pred,labels=[0,1]).ravel()
    threshold_table.append({'threshold':round(float(t),2),'FP':int(fp),'FN':int(fn),'cost':int(fp+fn),'precision':precision_score(y_train,pred,zero_division=0),'recall':recall_score(y_train,pred,zero_division=0)})
threshold_df=pd.DataFrame(threshold_table)
display(threshold_df)
chosen=float(threshold_df.loc[threshold_df['cost'].idxmin(),'threshold'])
print('Выбранный порог:',round(chosen,2))
plt.plot(threshold_df['threshold'],threshold_df['cost'],marker='o');plt.axvline(chosen,color='red',ls='--');plt.title('Стоимость ошибок на train (OOF)');plt.xlabel('Порог');plt.ylabel('FP + FN');plt.show()

## 6. Финальный тест: accuracy и confusion matrix

In [ ]:
best_model.fit(X_train,y_train)
proba=best_model.predict_proba(X_test)[:,1]
pred=(proba>=chosen).astype(int)
print('Модель:',best_name,'Порог:',round(chosen,2))
print('Accuracy:',round(accuracy_score(y_test,pred),4),'F1:',round(f1_score(y_test,pred),4),'ROC-AUC:',round(roc_auc_score(y_test,proba),4))
print(classification_report(y_test,pred,target_names=['<=50K','>50K'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['<=50K','>50K'],cmap='Blues');plt.title('Confusion matrix — тест');plt.show()
tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
print(f'Из людей с доходом >50K модель правильно нашла {tp} из {tp+fn} ({tp/(tp+fn):.1%}). Пропустила: {fn}.')
print(f'Ошибочно предложила карту людям с доходом <=50K: {fp}.')

## 7. Продвинутый: ошибки по группам
Измеряем FNR и FPR по группам с достаточным количеством наблюдений. Это диагностика различий, а не разрешение использовать чувствительные атрибуты для реального кредитования.

In [ ]:
audit=X_test[['sex','race','age']].copy()
audit['actual']=y_test.to_numpy();audit['pred']=pred
audit['age_group']=pd.cut(audit['age'],bins=[0,30,45,60,120],labels=['<=30','31-45','46-60','61+'])
for group_col in ['sex','race','age_group']:
    records=[]
    for group,g in audit.groupby(group_col,observed=True):
        if len(g)<30:continue
        tn_,fp_,fn_,tp_=confusion_matrix(g['actual'],g['pred'],labels=[0,1]).ravel()
        records.append({'group':str(group),'n':len(g),'FNR':round(fn_/(fn_+tp_),3) if fn_+tp_ else np.nan,'FPR':round(fp_/(fp_+tn_),3) if fp_+tn_ else np.nan,'accuracy':round(accuracy_score(g['actual'],g['pred']),3)})
    print('\nГруппа:',group_col);display(pd.DataFrame(records).sort_values('n',ascending=False))

## 8. Дисбаланс: два способа и сравнение
Сравним обычную Logistic Regression и `class_weight="balanced"` с одинаковой CV. Второй метод — настройка порога на train (выше). Метрики могут ухудшаться при повышении recall, поэтому оцениваем обе стороны ошибки.

In [ ]:
balance_models={
 'LogReg обычная':Pipeline([('prep',preprocessor()),('model',LogisticRegression(max_iter=1500,random_state=42))]),
 'LogReg class_weight=balanced':Pipeline([('prep',preprocessor()),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=42))])
}
comparison=[]
for name,model in balance_models.items():
    s=cross_validate(model,X_train,y_train,cv=cv,scoring={'precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'},n_jobs=-1)
    comparison.append({'method':name,**{m:round(s['test_'+m].mean(),4) for m in ['precision','recall','f1','roc_auc']}})
display(pd.DataFrame(comparison))

## 9. Заключение
- Пропуски `?` преобразованы в NaN и заполнены внутри Pipeline, что предотвращает утечку.
- Удалены дублирующий код образования и вес переписи.
- Кандидаты сравнивались на 5 стратифицированных фолдах train.
- Порог подобран по out-of-fold вероятностям, затем проведена однократная оценка на test.
- При равных условных стоимостях FP и FN минимизировали FP+FN; при других бизнес-стоимостях порог нужно выбирать заново.
- Анализ FNR/FPR по группам нужен для выявления различий и рисков дискриминации.
- Ограничения: старые переписные данные США, доход не равен кредитоспособности; для реальных решений необходимы правовая экспертиза, проверка качества и человеческий контроль.